In [ ]:
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'pyproject.toml').is_file() and (p / 'human_validation').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Run this notebook from inside the repository.')
FORM_DATA = ROOT / 'human_validation/events/data'
REPORTS = ROOT / 'human_validation/events/results/new_forms'
REPORTS.mkdir(parents=True, exist_ok=True)

import pandas as pd
import numpy as np
import json

# FILE CONFIGURATION — replace paths for each new form; no fixed filename/size is required.
FORM_NAME = 'FV - X'
JSON_FILE = FORM_DATA / 'sampled_events_seed_X.json'
HUMAN_CSV = FORM_DATA / 'FV - X.csv'
JUDGE_CSV = ROOT / 'experiments/events/results/tribunal_judgments.csv'
ORIGINAL_DATASET = ROOT / 'experiments/events/results/forensic_atomic_tribunal_validated.csv'
AGGREGATE_CSV = FORM_DATA / 'new_form_comparison.csv'
STATISTICS_JSON = FORM_DATA / 'new_form_statistics.json'
DASHBOARD_PATH = REPORTS / 'validation_dashboard.png'
COMPARISON_FILES = None  # set an explicit list of Paths, or use COMPARISON_GLOB
COMPARISON_GLOB = 'comparison_results_*.csv'
# Form exports contain one timestamp then score/vote/reason for each sampled event.
RESPONSE_START_COLUMN = 1
RESPONSE_COLUMNS_PER_EVENT = 3
SCORE_OFFSET = 0
DECISION_OFFSET = 1
HUMAN_SCORE_MULTIPLIER = 1  # 10 for raw 0–10 scores, 1 for 0–100
# Legacy logs used source row positions. Enable ONLY for a dataset whose ordering
# is known to match that log; preserve supplied IDs by default.
USE_DATASET_ROW_INDEX = False

# Columns to find the exact row in the original dataset
BRIDGE_COLS = ['event', 'xIntent', 'xEffect']  # may include original_row_id
if 'event' not in BRIDGE_COLS:
    raise ValueError('Include event in BRIDGE_COLS for matching to the judge CSV')

with open(JSON_FILE, 'r', encoding='utf-8') as f:
    sample_data = json.load(f)

if not sample_data.get('events'):
    raise ValueError('Sample JSON must contain a non-empty events list')
events_data = []
for e in sample_data['events']:
    # Get the info needed as a bridge
    event_info = {col: e.get(col, '') for col in BRIDGE_COLS}
    events_data.append(event_info)

print(f"Loaded {len(events_data)} events from JSON.")

# HUMAN RESPONSES PARSING
human_df = pd.read_csv(HUMAN_CSV)
if human_df.empty:
    raise ValueError('The form response CSV contains no responses')
required_response_columns = RESPONSE_START_COLUMN + (len(events_data) - 1) * RESPONSE_COLUMNS_PER_EVENT + max(SCORE_OFFSET, DECISION_OFFSET) + 1
if len(human_df.columns) < required_response_columns:
    raise ValueError('Response columns do not match the configured layout and sampled events')
records = []

for respondent_idx, row in human_df.iterrows():
    for i, event_info in enumerate(events_data):
        # Read the configured score and decision columns for sampled event i.
        base_col = RESPONSE_START_COLUMN + (i * RESPONSE_COLUMNS_PER_EVENT)

        if base_col + max(SCORE_OFFSET, DECISION_OFFSET) < len(human_df.columns):
            score_val = row.iloc[base_col + SCORE_OFFSET]
            decision_val = str(row.iloc[base_col + DECISION_OFFSET]).upper()

            record = event_info.copy()
            record.update({
                'respondent': respondent_idx,
                'human_score': float(score_val) * HUMAN_SCORE_MULTIPLIER if pd.notna(score_val) else np.nan,
                'human_approved_int': 1 if 'APPROVE' in decision_val else 0
            })
            records.append(record)

human_parsed = pd.DataFrame(records)
if 'original_row_id' in BRIDGE_COLS:
    human_parsed['original_row_id'] = human_parsed['original_row_id'].astype('string')

# Aggregate by calculating human means for each unique event
human_agg = human_parsed.groupby(BRIDGE_COLS).agg({
    'human_score': 'mean',
    'human_approved_int': lambda x: (x.sum() / len(x)) * 100
}).reset_index()

human_agg.rename(columns={'human_approved_int': 'human_approval_pct'}, inplace=True)
print("Human parsing and aggregation completed.")

# RETRIEVE 'original_row_id' FROM ORIGINAL DATASET
original_df = pd.read_csv(ORIGINAL_DATASET, dtype={'original_row_id': 'string'})
if USE_DATASET_ROW_INDEX:
    original_df['original_row_id'] = original_df.index.astype(str)
elif 'original_row_id' not in original_df:
    raise ValueError('Dataset needs original_row_id; use row positions only for a verified legacy log')
# Ensure there are no blank spaces ruining the match
for col in BRIDGE_COLS:
    human_agg[col] = human_agg[col].astype(str).str.strip()
    original_df[col] = original_df[col].astype(str).str.strip()

if original_df.duplicated(BRIDGE_COLS).any():
    raise ValueError('Ambiguous dataset bridge; configure BRIDGE_COLS to identify each sampled event uniquely')

# Retrieve the original ID by merging
human_with_id = pd.merge(human_agg, original_df[list(dict.fromkeys(BRIDGE_COLS + ['original_row_id']))], on=BRIDGE_COLS, how='left', validate='1:1')
if human_with_id['original_row_id'].isna().any():
    raise ValueError('Some sampled events do not match the configured original dataset')
print(f"Found {len(human_with_id)} exact original_row_ids from the original dataset.")

# MERGE WITH LLM JUDGES LOG
judge_df = pd.read_csv(JUDGE_CSV, dtype={'original_row_id': 'string'})
judge_df['event'] = judge_df['event'].astype('string').str.strip()
judge_df['score_avg_all'] = pd.to_numeric(judge_df['score_avg_all'], errors='coerce')
judge_keys = ['original_row_id', 'event']
if judge_df.duplicated(judge_keys).any():
    raise ValueError('The judge CSV has ambiguous event keys; select a unique evaluation run')

# Use the unique ID to merge humans and LLM
comparison = pd.merge(human_with_id, judge_df, on=judge_keys, how='left', validate='m:1', suffixes=('_human', '_llm'))
if comparison['status'].isna().any():
    raise ValueError('Some sampled events have no LLM judgment on original_row_id and event')
comparison['status'] = comparison['status'].astype('string').str.strip().str.upper()
if not comparison['status'].isin(['APPROVED', 'REJECTED']).all():
    raise ValueError('Judge status must be APPROVED or REJECTED')

# Calculate comparison metrics
comparison['form_id'] = FORM_NAME
comparison['human_is_approved'] = (comparison['human_approval_pct'] >= 50) & (comparison['human_score'] >= 50)
comparison['llm_is_approved'] = comparison['status'] == 'APPROVED'
comparison['score_diff_abs'] = abs(comparison['human_score'] - comparison['score_avg_all'])
comparison['decision_agreement'] = comparison['human_is_approved'] == comparison['llm_is_approved']

print(f"Final merge with judges completed: {len(comparison)} merged events.")

# EXPORT
# Select only important columns for future statistics
export_cols = [
    'form_id', 'original_row_id', 'event',
    'human_score', 'human_approval_pct', 'human_is_approved',
    'score_avg_all', 'status', 'llm_is_approved',
    'score_diff_abs', 'decision_agreement'
]

final_export = comparison[export_cols]

# Keep the form label in the CSV; sanitize separators for its report filename.
report_name = FORM_NAME.replace('/', '_').replace('\\', '_')
export_filename = REPORTS / f'comparison_results_{report_name}.csv'
final_export.to_csv(export_filename, index=False)

print(f"Data saved in: {export_filename}")

In [ ]:
import pandas as pd
from scipy.stats import pearsonr, spearmanr
import json

# Find all generated files matching the pattern
file_list = ([ROOT / Path(path) for path in COMPARISON_FILES]
             if COMPARISON_FILES is not None else sorted(REPORTS.glob(COMPARISON_GLOB)))
if not file_list:
    raise ValueError('No comparison files selected; configure COMPARISON_FILES or COMPARISON_GLOB')

print(f"Found {len(file_list)} files to merge:")
for f in file_list:
    print(f"  - {f}")

# Load and concatenate all dataframes into one
df_list = []
for file in file_list:
    df = pd.read_csv(file, dtype={'original_row_id': 'string'})
    if df.empty:
        raise ValueError(f'Comparison file contains no evaluations: {file}')
    df_list.append(df)

global_df = pd.concat(df_list, ignore_index=True)
print(f"\nMerge completed, total analyzed events: {len(global_df)}")

# Save the global dataset for future analysis
global_csv_name = AGGREGATE_CSV
global_csv_name.parent.mkdir(parents=True, exist_ok=True)
global_df.to_csv(global_csv_name, index=False)
print(f"Global dataset saved in: '{global_csv_name}'")

# Remove any rows with null scores before correlations
valid_scores = global_df.dropna(subset=['human_score', 'score_avg_all'])

# Calculate global correlation between humans and LLM
if len(valid_scores) >= 2 and valid_scores[['human_score', 'score_avg_all']].nunique().gt(1).all():
    pearson_corr, p_value_p = pearsonr(valid_scores['human_score'], valid_scores['score_avg_all'])
    spearman_corr, p_value_s = spearmanr(valid_scores['human_score'], valid_scores['score_avg_all'])
else:
    pearson_corr = p_value_p = spearman_corr = p_value_s = float('nan')

print("\nGLOBAL STATISTICS (ALL FORMS)")
print(f"Total Evaluated Events:           {len(global_df)}")
print(f"Human Approval Rate:              {(global_df['human_is_approved'].mean() * 100):.1f}%")
print(f"LLM Approval Rate:                {(global_df['llm_is_approved'].mean() * 100):.1f}%")
print(f"Decision Agreement (Appr/Rej):    {(global_df['decision_agreement'].mean() * 100):.1f}%")
print(f"Global Mean Score (Humans):       {global_df['human_score'].mean():.1f} / 100")
print(f"Global Mean Score (LLM):          {global_df['score_avg_all'].mean():.1f} / 100")
print(f"Mean Absolute Score Difference:   {global_df['score_diff_abs'].mean():.1f} points")
print(f"Pearson Correlation (r):          {pearson_corr:.3f} (p-value: {p_value_p:.4f})")
print(f"Spearman Correlation (rho):       {spearman_corr:.3f} (p-value: {p_value_s:.4f})")

stats_dict = {
    "total_events": len(global_df),
    "human_approval_pct": round(global_df['human_is_approved'].mean() * 100, 1),
    "llm_approval_pct": round(global_df['llm_is_approved'].mean() * 100, 1),
    "decision_agreement_pct": round(global_df['decision_agreement'].mean() * 100, 1),
    "mean_human_score": round(global_df['human_score'].mean(), 1),
    "mean_llm_score": round(global_df['score_avg_all'].mean(), 1),
    "mean_absolute_diff": round(global_df['score_diff_abs'].mean(), 1),
    "pearson_r": round(pearson_corr, 3),
    "spearman_rho": round(spearman_corr, 3)
}

STATISTICS_JSON.parent.mkdir(parents=True, exist_ok=True)
with open(STATISTICS_JSON, 'w') as f:
    json.dump(stats_dict, f, indent=4)
print(f"\nStatistics exported to {STATISTICS_JSON}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use("default")
import seaborn as sns
from scipy.stats import pearsonr, spearmanr

# Load global data
df = pd.read_csv(AGGREGATE_CSV)
# Enforce approval logic
df['human_is_approved'] = (df['human_approval_pct'] >= 50) & (df['human_score'] >= 50)

# Calculate descriptive statistics
total = len(df)
agreement_rate = df['decision_agreement'].mean() * 100
human_app_rate = df['human_is_approved'].mean() * 100
llm_app_rate = df['llm_is_approved'].mean() * 100
avg_human_score = df['human_score'].mean()
avg_llm_score = df['score_avg_all'].mean()
mae = df['score_diff_abs'].mean()

# Calculate correlations (excluding NaNs)
valid = df.dropna(subset=['human_score', 'score_avg_all'])
if len(valid) >= 2 and valid[['human_score', 'score_avg_all']].nunique().gt(1).all():
    p_r, p_val_p = pearsonr(valid['human_score'], valid['score_avg_all'])
    s_r, p_val_s = spearmanr(valid['human_score'], valid['score_avg_all'])
else:
    p_r = p_val_p = s_r = p_val_s = float('nan')

# Visual Dashboard Setup
sns.set_style("whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
plt.suptitle(f"Global Analysis: Human vs LLM Validation ({total} Events)", fontsize=18, fontweight='bold', y=1.02)

# Pie Chart (Decision Agreement)
agreement_counts = df['decision_agreement'].value_counts().reindex([True, False], fill_value=0)
axes[0, 0].pie(agreement_counts, labels=['Agreement', 'Disagreement'], autopct='%1.1f%%',
               colors=['#4CAF50', '#F44336'], startangle=140, explode=(0.1, 0), shadow=True)
axes[0, 0].set_title('Decision Agreement (Approve/Reject)', fontweight='bold', fontsize=14)

# Bar Chart (Approval Rates)
rates = [human_app_rate, llm_app_rate]
labels = ['Humans', 'LLM Judge']
sns.barplot(x=labels, y=rates, palette=['#2196F3', '#FF9800'], ax=axes[0, 1], edgecolor='black')
axes[0, 1].set_ylim(0, 110)
axes[0, 1].set_ylabel('Approval Percentage (%)', fontsize=12)
axes[0, 1].set_title('Approval Rates Comparison', fontweight='bold', fontsize=14)
for i, v in enumerate(rates):
    axes[0, 1].text(i, v + 2, f"{v:.1f}%", ha='center', fontweight='bold', fontsize=12)

# Distribution (Score Density)
for column, label, color in [('human_score', 'Humans', '#2196F3'), ('score_avg_all', 'LLM', '#FF9800')]:
    values = df[column].dropna()
    if len(values) >= 2 and values.nunique() > 1:
        sns.kdeplot(values, ax=axes[1, 0], fill=True, label=label, color=color, alpha=0.5)
    elif not values.empty:
        axes[1, 0].axvline(values.iloc[0], label=label, color=color)
axes[1, 0].set_xlabel('Consistency Score (0-100)', fontsize=12)
axes[1, 0].set_title('Qualitative Score Distribution', fontweight='bold', fontsize=14)
axes[1, 0].legend(loc='upper left')

# Scatter Plot + Regression (Correlation)
sns.regplot(x='human_score', y='score_avg_all', data=df, ax=axes[1, 1],
            scatter_kws={'alpha':0.4, 'color':'purple', 's':50},
            line_kws={'color':'red', 'label': f'Pearson r: {p_r:.2f}'},
            fit_reg=len(valid) >= 2 and valid['human_score'].nunique() > 1)
axes[1, 1].set_title('Correlation and Trend Line', fontweight='bold', fontsize=14)
axes[1, 1].set_xlabel('Human Score', fontsize=12)
axes[1, 1].set_ylabel('LLM Score', fontsize=12)
axes[1, 1].legend()

# Finalization
plt.tight_layout()
DASHBOARD_PATH.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(DASHBOARD_PATH, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print("STATISTICAL SUMMARY FOR THESIS")
print(f"Analyzed Sample:          {total} events")
print(f"Decision Agreement:       {agreement_rate:.2f}%")
print(f"Human Approval Rate:      {human_app_rate:.1f}%")
print(f"LLM Approval Rate:        {llm_app_rate:.1f}%")
print(f"Mean Human Score:         {avg_human_score:.2f} / 100")
print(f"Mean LLM Score:           {avg_llm_score:.2f} / 100")
print(f"MAE (Mean Absolute Error):{mae:.2f} points")
print(f"Pearson r:                {p_r:.3f} (p-value: {p_val_p:.4e})")
print(f"Spearman rho:             {s_r:.3f} (p-value: {p_val_s:.4e})")

In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix

df = pd.read_csv(AGGREGATE_CSV)
valid_df = df.dropna(subset=['human_is_approved', 'llm_is_approved'])

# Human = Truth (y_true), LLM = Prediction (y_pred)
# True = 1, False = 0
y_true = valid_df['human_is_approved'].astype(int)
y_pred = valid_df['llm_is_approved'].astype(int)

# Calculate Confusion Matrix
tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

print(f"True Positive (TP) [Both OK]: {tp}")
print(f"False Positive (FP) [Human NO, LLM OK]: {fp}")
print(f"False Negative (FN) [Human OK, LLM NO]: {fn}")
print(f"True Negative (TN) [Both NO]: {tn}")